# Explainability: Grad-CAM and AAL2 region ranking

Runs the native 3D Grad-CAM (`multimodal_ad.models.gradcam`) on a tiny synthetic volume/model, then the AAL2 region-ranking utility (`multimodal_ad.models.regions`) against the real AAL2 atlas bundled at the repository root (`atlas.nii.gz` + `AAL2_Atlas_Labels.csv`), using a synthetic Grad-CAM heatmap so this stays OASIS-free. Requires the `model` extra: `uv sync --extra model`.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

from multimodal_ad.models.architecture import Cnn3DConfig, build_3d_cnn
from multimodal_ad.models.gradcam import (
    last_conv_layer_name,
    make_gradcam_heatmap,
    unwrap_output_activation,
)
from multimodal_ad.models.regions import (
    ATLAS_PLACEMENT,
    HEATMAP_PLACEMENT,
    load_atlas,
    load_region_labels,
    pad_to_frame,
    rank_regions,
)

## Native Grad-CAM on a tiny synthetic volume

Same reduced-filter model shape as `02-tiny-model-workflow.ipynb`. Grad-CAM needs raw logits, not a squashed sigmoid output, hence `unwrap_output_activation`.

In [ ]:
SIZE = 48
rng = np.random.default_rng(1234)

config = Cnn3DConfig(
    width=SIZE, height=SIZE, depth=SIZE, filters=(4, 4, 8, 8), dense_units=16,
    name="tiny-gradcam",
)
model = build_3d_cnn(config)
unwrap_output_activation(model)

layer_name = last_conv_layer_name(model)
volume = rng.random((1, SIZE, SIZE, SIZE, 1)).astype("float32")
heatmap = make_gradcam_heatmap(volume, model, layer_name)
heatmap.shape, float(heatmap.min()), float(heatmap.max())

## AAL2 region ranking: the bundled real atlas

`rank_regions` needs an AAL2 atlas volume and its region labels. Both are bundled at the repository root: `atlas.nii.gz` (the real `(91, 109, 91)` AAL2 volume) and `AAL2_Atlas_Labels.csv` (region name -> intensity mapping). `load_atlas` and `load_region_labels` load them; `pad_to_frame` places the atlas into the common `(128, 128, 128)` frame the legacy notebook used, at the retained `ATLAS_PLACEMENT` offset.

To keep this notebook OASIS-free, the *heatmap* side stays synthetic: a small, deterministic random array standing in for a real Grad-CAM output, padded into the same common frame at `HEATMAP_PLACEMENT`.

In [ ]:
REPO_ROOT = Path.cwd().parent  # notebook kernels cwd to notebooks/

atlas = load_atlas(REPO_ROOT / "atlas.nii.gz")
atlas_frame = pad_to_frame(atlas, ATLAS_PLACEMENT)
region_labels = load_region_labels(str(REPO_ROOT / "AAL2_Atlas_Labels.csv"))

heatmap_rng = np.random.default_rng(42)
synthetic_heatmap = heatmap_rng.random((128, 128, 50)).astype(np.float64)
heatmap_frame = pad_to_frame(synthetic_heatmap, HEATMAP_PLACEMENT)

ranking = rank_regions(atlas_frame, region_labels, {"synthetic": heatmap_frame})
ranking.head()

## Legacy `mean` vs. corrected `region mean`

`rank_regions` reports two different mean columns (see `multimodal_ad.models.regions` module docstring for the full derivation):

- `"{name} mean"`: the **legacy notebook's exact formula** (`exploration.ipynb`), region sum divided by the *entire common-frame voxel count* (here, all `128 * 128 * 128` voxels), not the region's own size. Kept for reproducing the published paper's region tables; it shrinks toward zero for small regions purely from a huge, mostly-zero denominator, not from lower Grad-CAM importance.
- `"{name} region mean"`: the **corrected**, size-comparable mean, region sum divided by the region's *own* voxel count. This is what "mean Grad-CAM in region X" should mean, and is what new analysis should use.

In [ ]:
ranking.sort_values("synthetic region mean", ascending=False)[
    ["part", "synthetic mean", "synthetic region mean"]
].head(10)

## Next steps

- Legacy region-ranking notebook: `notebooks/legacy/exploration-region-ranking.ipynb`.
- Open ambiguities in the atlas/heatmap spatial alignment are documented in `docs/legacy-notebooks-inventory.md`.